# Model Performance: Results Presentation

Structure:
1. **Table 1** — Main DiD results (full sample)
2. **Figure 1** — Rolling-window DiD over time
3. **Table 2** — Sub-period DiD breakdown
4. **Appendix** — Scatter grid, Spearman, directional accuracy, country-level tables

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.facecolor': 'white',
})

In [2]:
# ============================================================
# LOAD DATA & DEFINE GROUPS
# ============================================================
models = {}
for name in ['M0', 'M1', 'M2']:
    df = pd.read_csv(f'../output/results/{name}_results.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['country', 'date']).reset_index(drop=True)
    models[name] = df

group_map = {
    'Saudi Arabia': 'Oil Exporter',
    'Abu Dhabi': 'Oil Exporter',
    'Dubai': 'Oil Exporter',
    'Qatar': 'Oil Exporter',
    'Colombia': 'Oil Exporter',
    'Mexico': 'Oil Exporter',
    'Brazil': 'Oil Exporter',
    'Egypt': 'Oil Exporter',
    'Malaysia': 'Oil Exporter',
    'Indonesia': 'Control',
    'Philippines': 'Control',
    'Turkey': 'Control',
    'Chile': 'Control',
    'China': 'Control',
    'South Africa': 'Control',
    'South Korea': 'Control',
    'Thailand': 'Control',
}

EXPORTER_COUNTRIES = [c for c, g in group_map.items() if g == 'Oil Exporter']
CONTROL_COUNTRIES  = [c for c, g in group_map.items() if g == 'Control']

PERIODS = {
    'Full Sample':  ('2015-01-01', '2024-12-31'),
    '2015–2016':    ('2015-01-01', '2016-12-31'),
    '2017–2019':    ('2017-01-01', '2019-12-31'),
    '2020–2021':    ('2020-01-01', '2021-12-31'),
    '2022–2024':    ('2022-01-01', '2024-12-31'),
}

HORIZONS = {'1w': 1, '1m': 4, '3m': 12}

print(f'Exporters: {len(EXPORTER_COUNTRIES)}, Controls: {len(CONTROL_COUNTRIES)}')
for mname, df in models.items():
    print(f'{mname}: {df["country"].nunique()} countries, {len(df)} rows, '
          f'{df["date"].min().date()} to {df["date"].max().date()}')

Exporters: 9, Controls: 8
M0: 17 countries, 9758 rows, 2014-01-05 to 2024-12-29
M1: 17 countries, 9758 rows, 2014-01-05 to 2024-12-29
M2: 17 countries, 9758 rows, 2014-01-05 to 2024-12-29


In [3]:
# ============================================================
# CORE UTILITIES
# ============================================================

def nw_pvalue(x, y, nlags=None):
    """Correlation t-test with Newey-West HAC standard errors."""
    n = len(x)
    if nlags is None:
        nlags = int(n ** (1/3))
    xd = x - x.mean()
    yd = y - y.mean()
    xy = xd * yd
    gamma0 = np.var(xy, ddof=1)
    nw = gamma0
    for j in range(1, nlags + 1):
        w = 1 - j / (nlags + 1)
        nw += 2 * w * np.cov(xy[j:], xy[:-j], ddof=1)[0, 1]
    t = np.mean(xy) / np.sqrt(nw / n)
    return 2 * (1 - stats.t.cdf(abs(t), df=n - 2))


def star(p):
    if pd.isna(p): return ''
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''


def compute_country_corr(df, dd_col='distance_to_distress', cds_col='cds_spread',
                         start=None, end=None):
    """
    Compute per-country correlations for all horizons.
    Returns DataFrame with one row per country.
    """
    if start:
        df = df[(df['date'] >= start) & (df['date'] <= end)]

    rows = []
    for country in sorted(df['country'].unique()):
        d = df[df['country'] == country].sort_values('date').copy()
        group = group_map.get(country, 'Control')
        row = {'Country': country, 'Group': group}

        for hlbl, h in HORIZONS.items():
            dx = d[dd_col].diff(h)
            dy = d[cds_col].diff(h)
            valid = dx.notna() & dy.notna()
            x, y = dx[valid].values, dy[valid].values

            if len(x) < 20:
                row[f'rho_{hlbl}'] = np.nan
                row[f'p_{hlbl}'] = np.nan
                row[f'rhoS_{hlbl}'] = np.nan
                row[f'Dir_{hlbl}'] = np.nan
                row[f'n_{hlbl}'] = len(x)
                continue

            rho, _ = stats.pearsonr(x, y)
            p_nw = nw_pvalue(x, y)
            rho_s, _ = stats.spearmanr(x, y)
            nz = (x != 0) & (y != 0)
            dir_acc = np.mean((x[nz] > 0) == (y[nz] < 0)) if nz.sum() > 0 else np.nan

            row[f'rho_{hlbl}'] = rho
            row[f'p_{hlbl}'] = p_nw
            row[f'rhoS_{hlbl}'] = rho_s
            row[f'Dir_{hlbl}'] = dir_acc
            row[f'n_{hlbl}'] = len(x)

        rows.append(row)
    return pd.DataFrame(rows)


def group_means(corr_df, metric='rho'):
    """Compute exporter mean, control mean, and DiD for each horizon."""
    out = {}
    for hlbl in HORIZONS:
        col = f'{metric}_{hlbl}'
        exp = corr_df[corr_df['Group'] == 'Oil Exporter'][col].dropna()
        ctrl = corr_df[corr_df['Group'] == 'Control'][col].dropna()
        out[hlbl] = {
            'exp_mean': exp.mean(),
            'ctrl_mean': ctrl.mean(),
            'did': exp.mean() - ctrl.mean(),
            'exp_n': len(exp),
            'ctrl_n': len(ctrl),
        }
    return out


def did_permutation_test(corr_df, hlbl, metric='rho', n_perm=5000, seed=42):
    """
    Permutation test for DiD: shuffle group labels, recompute DiD, get p-value.
    """
    rng = np.random.default_rng(seed)
    col = f'{metric}_{hlbl}'
    sub = corr_df[['Group', col]].dropna()
    values = sub[col].values
    groups = sub['Group'].values

    obs_did = values[groups == 'Oil Exporter'].mean() - values[groups == 'Control'].mean()

    null_dids = np.empty(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(groups)
        null_dids[i] = values[perm == 'Oil Exporter'].mean() - values[perm == 'Control'].mean()

    p = np.mean(np.abs(null_dids) >= np.abs(obs_did))
    return obs_did, p

In [4]:
# ============================================================
# PRECOMPUTE ALL CORRELATIONS
# ============================================================
# corr_results[model_name][period_label] = DataFrame
corr_results = {}
for mname, df in models.items():
    corr_results[mname] = {}
    for plabel, (start, end) in PERIODS.items():
        corr_results[mname][plabel] = compute_country_corr(df, start=start, end=end)

print('Precomputation done.')

Precomputation done.


---
## Table 1 — Main Results (Full Sample)

Pearson ρ between ΔDtD and ΔCDS at 1-week, 1-month, 3-month horizons.  
DiD = Exporter mean − Control mean. P-values from permutation test.

In [5]:
# ============================================================
# TABLE 1: MAIN DiD RESULTS (FULL SAMPLE)
# ============================================================

def build_main_table(corr_results, period='Full Sample'):
    """Build the compact DiD summary table."""
    rows = []

    for hlbl in HORIZONS:
        row = {'Horizon': hlbl}
        for mname in ['M0', 'M1', 'M2']:
            cdf = corr_results[mname][period]
            gm = group_means(cdf)

            # Permutation test on the DiD
            did_val, did_p = did_permutation_test(cdf, hlbl)

            row[f'{mname}_Exp'] = gm[hlbl]['exp_mean']
            row[f'{mname}_Ctrl'] = gm[hlbl]['ctrl_mean']
            row[f'{mname}_DiD'] = did_val
            row[f'{mname}_DiD_p'] = did_p
        rows.append(row)

    return pd.DataFrame(rows)


def display_main_table(tbl):
    """Pretty-print the main results table."""
    print(f"{'':>8}", end='')
    for mname in ['M0', 'M1', 'M2']:
        print(f'  |{mname:^30s}', end='')
    print()

    print(f'{"Horizon":>8}', end='')
    for _ in ['M0', 'M1', 'M2']:
        print(f'  |{"Exp":>9}{"Ctrl":>9}{"DiD":>12}', end='')
    print()
    print('—' * 104)

    for _, r in tbl.iterrows():
        print(f'{r["Horizon"]:>8}', end='')
        for mname in ['M0', 'M1', 'M2']:
            e = r[f'{mname}_Exp']
            c = r[f'{mname}_Ctrl']
            d = r[f'{mname}_DiD']
            p = r[f'{mname}_DiD_p']
            s = star(p)
            print(f'  |{e:>9.3f}{c:>9.3f}{d:>+9.3f}{s:<3s}', end='')
        print()

    print('—' * 104)
    print('Permutation test (5000 draws). * p<0.10, ** p<0.05, *** p<0.01')


tbl1 = build_main_table(corr_results)
display_main_table(tbl1)

          |              M0                |              M1                |              M2              
 Horizon  |      Exp     Ctrl         DiD  |      Exp     Ctrl         DiD  |      Exp     Ctrl         DiD
————————————————————————————————————————————————————————————————————————————————————————————————————————
      1w  |   -0.071   -0.055   -0.016     |   -0.203   -0.172   -0.031     |   -0.145   -0.123   -0.021   
      1m  |   -0.132   -0.090   -0.041     |   -0.301   -0.214   -0.087     |   -0.273   -0.217   -0.056   
      3m  |   -0.164   -0.176   +0.012     |   -0.387   -0.335   -0.052     |   -0.383   -0.333   -0.050   
————————————————————————————————————————————————————————————————————————————————————————————————————————
Permutation test (5000 draws). * p<0.10, ** p<0.05, *** p<0.01


---
## Figure 1 — Rolling-Window DiD

12-month rolling Pearson ρ(ΔDtD, ΔCDS) computed per country, then averaged within group.  
DiD = Exporter avg − Control avg. One line per model.

In [6]:
# ============================================================
# FIGURE 1: ROLLING-WINDOW DiD
# ============================================================

def compute_rolling_did(models, group_map, window=52, horizon=4,
                        dd_col='distance_to_distress', cds_col='cds_spread'):
    """
    For each model, compute rolling correlation per country,
    then average within group, then DiD.
    Returns dict of DataFrames indexed by date.
    """
    results = {}

    for mname, df in models.items():
        df = df[df['country'].isin(group_map.keys())].copy()
        df = df.sort_values(['country', 'date'])

        # Compute changes per country
        df['dd_chg'] = df.groupby('country')[dd_col].diff(horizon)
        df['cds_chg'] = df.groupby('country')[cds_col].diff(horizon)

        # Rolling correlation per country
        rolling_corrs = []
        for country in df['country'].unique():
            d = df[df['country'] == country][['date', 'dd_chg', 'cds_chg']].set_index('date')
            rc = d['dd_chg'].rolling(window, min_periods=int(window * 0.6)).corr(d['cds_chg'])
            rc = rc.rename(country).to_frame()
            rc['country'] = country
            rc['group'] = group_map[country]
            rolling_corrs.append(rc.reset_index())

        all_rc = pd.concat(rolling_corrs)
        all_rc.columns = ['date', 'rho', 'country', 'group']

        # Group means per date
        gm = all_rc.groupby(['date', 'group'])['rho'].mean().unstack('group')
        gm['DiD'] = gm.get('Oil Exporter', 0) - gm.get('Control', 0)
        results[mname] = gm

    return results


rolling = compute_rolling_did(models, group_map, window=52, horizon=4)

# --- Plot ---
MODEL_COLORS = {'M0': '#7f8c8d', 'M1': '#C0392B', 'M2': '#2980B9'}
MODEL_STYLES = {'M0': '--', 'M1': '-', 'M2': '-.'}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

# --- Top panel: group-level rolling rho ---
ax = axes[0]
# Show M0 exporter vs control as shaded context
for mname in ['M0']:
    gm = rolling[mname]
    ax.plot(gm.index, gm.get('Oil Exporter', np.nan),
            color=MODEL_COLORS[mname], ls='--', lw=1, alpha=0.6, label=f'{mname} Exporters')
    ax.plot(gm.index, gm.get('Control', np.nan),
            color=MODEL_COLORS[mname], ls=':', lw=1, alpha=0.6, label=f'{mname} Controls')

# Show M1 exporter vs control
for mname in ['M1']:
    gm = rolling[mname]
    ax.plot(gm.index, gm.get('Oil Exporter', np.nan),
            color=MODEL_COLORS[mname], ls='-', lw=1.8, label=f'{mname} Exporters')
    ax.plot(gm.index, gm.get('Control', np.nan),
            color=MODEL_COLORS[mname], ls=':', lw=1.2, label=f'{mname} Controls')

ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Mean Pearson ρ (ΔDtD, ΔCDS)')
ax.set_title('Rolling 12-Month Correlation: Exporters vs Controls', fontweight='bold')
ax.legend(fontsize=8, ncol=2, loc='lower left')
ax.grid(True, alpha=0.15)

# Period shading
PERIOD_COLORS = {
    '2015–2016': '#FADBD8', '2017–2019': '#D5F5E3',
    '2020–2021': '#FADBD8', '2022–2024': '#D5F5E3',
}
for ax_ in axes:
    for plabel, (s, e) in list(PERIODS.items())[1:]:  # skip Full Sample
        ax_.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                    alpha=0.08, color=PERIOD_COLORS.get(plabel, 'grey'))

# --- Bottom panel: DiD ---
ax = axes[1]
for mname in ['M0', 'M1', 'M2']:
    gm = rolling[mname]
    ax.plot(gm.index, gm['DiD'],
            color=MODEL_COLORS[mname], ls=MODEL_STYLES[mname],
            lw=1.8, label=mname)

ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('DiD (Exp − Ctrl)')
ax.set_xlabel('Date')
ax.legend(fontsize=9, ncol=3)
ax.grid(True, alpha=0.15)

fig.tight_layout()
fig.savefig('../output/figures/rolling_did.png', dpi=200, bbox_inches='tight')
plt.show()

ValueError: Length mismatch: Expected axis has 20 elements, new values have 4 elements

---
## Table 2 — Sub-Period Results

Same DiD structure as Table 1, broken down by regime.

In [ ]:
# ============================================================
# TABLE 2: SUB-PERIOD DiD
# ============================================================

def build_subperiod_table(corr_results):
    """One row per (period, horizon), columns per model."""
    rows = []
    sub_periods = {k: v for k, v in PERIODS.items() if k != 'Full Sample'}

    for plabel in sub_periods:
        for hlbl in HORIZONS:
            row = {'Period': plabel, 'Horizon': hlbl}
            for mname in ['M0', 'M1', 'M2']:
                cdf = corr_results[mname][plabel]
                gm = group_means(cdf)
                did_val, did_p = did_permutation_test(cdf, hlbl)

                row[f'{mname}_Exp'] = gm[hlbl]['exp_mean']
                row[f'{mname}_Ctrl'] = gm[hlbl]['ctrl_mean']
                row[f'{mname}_DiD'] = did_val
                row[f'{mname}_DiD_p'] = did_p
            rows.append(row)

    return pd.DataFrame(rows)


def display_subperiod_table(tbl):
    """Pretty-print sub-period results."""
    print(f'{"":>12}{"":>8}', end='')
    for mname in ['M0', 'M1', 'M2']:
        print(f'  |{mname:^30s}', end='')
    print()

    print(f'{"Period":>12}{"Hz":>8}', end='')
    for _ in ['M0', 'M1', 'M2']:
        print(f'  |{"Exp":>9}{"Ctrl":>9}{"DiD":>12}', end='')
    print()
    print('—' * 116)

    prev_period = None
    for _, r in tbl.iterrows():
        if r['Period'] != prev_period and prev_period is not None:
            print('—' * 116)
        prev_period = r['Period']

        print(f'{r["Period"]:>12}{r["Horizon"]:>8}', end='')
        for mname in ['M0', 'M1', 'M2']:
            e = r[f'{mname}_Exp']
            c = r[f'{mname}_Ctrl']
            d = r[f'{mname}_DiD']
            p = r[f'{mname}_DiD_p']
            s = star(p)
            if pd.notna(e):
                print(f'  |{e:>9.3f}{c:>9.3f}{d:>+9.3f}{s:<3s}', end='')
            else:
                print(f'  |{"—":>9}{"—":>9}{"—":>12}', end='')
        print()

    print('—' * 116)
    print('Permutation test (5000 draws). * p<0.10, ** p<0.05, *** p<0.01')


tbl2 = build_subperiod_table(corr_results)
display_subperiod_table(tbl2)

---
## Table 2b — DiD Improvement: M1 vs M0, M2 vs M0

Shows the *change* in DiD from adding oil information.  
This is the difference-in-differences-in-differences: does the DiD gap widen under M1/M2?

In [ ]:
# ============================================================
# TABLE 2b: DiD IMPROVEMENT OVER BASELINE
# ============================================================

def build_improvement_table(corr_results):
    """For each period/horizon: DiD(M1)-DiD(M0) and DiD(M2)-DiD(M0)."""
    rows = []
    for plabel in PERIODS:
        for hlbl in HORIZONS:
            row = {'Period': plabel, 'Horizon': hlbl}

            did_m0 = group_means(corr_results['M0'][plabel])
            did_m1 = group_means(corr_results['M1'][plabel])
            did_m2 = group_means(corr_results['M2'][plabel])

            d0 = did_m0[hlbl]['did']
            d1 = did_m1[hlbl]['did']
            d2 = did_m2[hlbl]['did']

            row['M0_DiD'] = d0
            row['M1_DiD'] = d1
            row['M2_DiD'] = d2
            row['M1_impr'] = d1 - d0
            row['M2_impr'] = d2 - d0
            rows.append(row)

    return pd.DataFrame(rows)


tbl2b = build_improvement_table(corr_results)

print(f'{"Period":>12}{"Hz":>6}  |{"M0 DiD":>10}{"M1 DiD":>10}{"M2 DiD":>10}  |'
      f'{"M1−M0":>10}{"M2−M0":>10}')
print('—' * 78)
prev = None
for _, r in tbl2b.iterrows():
    if r['Period'] != prev and prev is not None:
        print('—' * 78)
    prev = r['Period']
    print(f'{r["Period"]:>12}{r["Horizon"]:>6}  |'
          f'{r["M0_DiD"]:>+10.4f}{r["M1_DiD"]:>+10.4f}{r["M2_DiD"]:>+10.4f}  |'
          f'{r["M1_impr"]:>+10.4f}{r["M2_impr"]:>+10.4f}')

---
## Figure 2 — Sub-Period DiD Bar Chart

Visual companion to Table 2. Groups bars by period, one color per model.

In [ ]:
# ============================================================
# FIGURE 2: SUB-PERIOD DiD BAR CHART (1m horizon)
# ============================================================

target_horizon = '1m'  # <-- change if you want a different horizon

sub_periods = [k for k in PERIODS if k != 'Full Sample']
x = np.arange(len(sub_periods))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, mname in enumerate(['M0', 'M1', 'M2']):
    dids = []
    for plabel in sub_periods:
        gm = group_means(corr_results[mname][plabel])
        dids.append(gm[target_horizon]['did'])

    bars = ax.bar(x + i * width, dids, width,
                  label=mname, color=MODEL_COLORS[mname], alpha=0.85,
                  edgecolor='white', linewidth=0.5)

    # Add value labels
    for bar, val in zip(bars, dids):
        ypos = bar.get_height()
        va = 'bottom' if ypos >= 0 else 'top'
        ax.text(bar.get_x() + bar.get_width() / 2, ypos,
                f'{val:+.3f}', ha='center', va=va, fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(sub_periods)
ax.axhline(0, color='black', lw=0.6)
ax.set_ylabel(f'DiD: Exporter ρ − Control ρ ({target_horizon} horizon)')
ax.set_title(f'DiD by Sub-Period ({target_horizon} Horizon)', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.15)

fig.tight_layout()
fig.savefig('../output/figures/subperiod_did_bar.png', dpi=200, bbox_inches='tight')
plt.show()

---
## M1 vs M0, then M2 vs M0 — Sequential Presentation

Present results incrementally: what does M1 add? Then what does M2 add?

In [ ]:
# ============================================================
# SEQUENTIAL COMPARISON: M1 vs M0, THEN M2 vs M0
# ============================================================

def print_pairwise_comparison(corr_results, mname_new, mname_base='M0',
                               period='Full Sample'):
    """Show how a new model changes exporter/control correlations vs baseline."""
    base = corr_results[mname_base][period]
    new = corr_results[mname_new][period]

    print(f'\n{mname_new} vs {mname_base} ({period})')
    print(f'{"Horizon":>8}{"":>4}  |{mname_base + " Exp":>10}{mname_new + " Exp":>10}{"Δ Exp":>10}'
          f'  |{mname_base + " Ctrl":>10}{mname_new + " Ctrl":>10}{"Δ Ctrl":>10}'
          f'  |{"ΔDiD":>10}')
    print('—' * 95)

    for hlbl in HORIZONS:
        gm_b = group_means(base)
        gm_n = group_means(new)

        eb = gm_b[hlbl]['exp_mean']
        en = gm_n[hlbl]['exp_mean']
        cb = gm_b[hlbl]['ctrl_mean']
        cn = gm_n[hlbl]['ctrl_mean']

        d_exp = en - eb
        d_ctrl = cn - cb
        d_did = (en - cn) - (eb - cb)

        print(f'{hlbl:>8}{"":>4}  |{eb:>10.4f}{en:>10.4f}{d_exp:>+10.4f}'
              f'  |{cb:>10.4f}{cn:>10.4f}{d_ctrl:>+10.4f}'
              f'  |{d_did:>+10.4f}')


print_pairwise_comparison(corr_results, 'M1')
print_pairwise_comparison(corr_results, 'M2')

---
# Appendix

## A1 — Scatter Grid (z-scored DtD vs CDS, 4 periods × 3 models)

In [ ]:
# ============================================================
# APPENDIX FIGURE: 4x3 SCATTER GRID
# ============================================================

SCATTER_COLORS = {'Oil Exporters': '#C0392B', 'Controls': '#2C5F8A'}

def plot_d2_scatter_grid(models, group_map,
                         dd_col='distance_to_distress',
                         cds_col='cds_spread',
                         periods=None, save_path=None):
    if periods is None:
        periods = {k: v for k, v in PERIODS.items() if k != 'Full Sample'}

    nrows = len(periods)
    ncols = len(['M0', 'M1', 'M2'])
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))

    for row, (period_label, (start, end)) in enumerate(periods.items()):
        for col, mname in enumerate(['M0', 'M1', 'M2']):
            ax = axes[row, col]
            df = models[mname].copy()
            df = df[df['country'].isin(group_map.keys())]
            df = df[(df['date'] >= start) & (df['date'] <= end)]
            df = df.dropna(subset=[dd_col, cds_col])
            df = df[(df[dd_col] > 0) & (df[cds_col] > 0)]

            if len(df) < 20:
                ax.text(0.5, 0.5, 'Insufficient data', ha='center',
                        va='center', transform=ax.transAxes, fontsize=9)
                continue

            # Z-score per country within this period
            for c in [dd_col, cds_col]:
                df[f'{c}_z'] = df.groupby('country')[c].transform(
                    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
                )

            df['group'] = df['country'].map(group_map).map(
                lambda g: 'Oil Exporters' if g == 'Oil Exporter' else 'Controls'
            )

            for group, color in SCATTER_COLORS.items():
                g = df[df['group'] == group]
                x = g[f'{dd_col}_z'].values
                y = g[f'{cds_col}_z'].values

                ax.scatter(x, y, s=6, alpha=0.25, color=color,
                           edgecolors='none', label=group if row == 0 else None)

                mask = np.isfinite(x) & np.isfinite(y)
                if mask.sum() > 20:
                    coeffs = np.polyfit(x[mask], y[mask], 2)
                    x_fit = np.linspace(np.percentile(x[mask], 2),
                                        np.percentile(x[mask], 98), 200)
                    y_fit = np.polyval(coeffs, x_fit)
                    ax.plot(x_fit, y_fit, color=color, lw=2.5, zorder=5)

                    y_pred = np.polyval(coeffs, x[mask])
                    ss_res = np.sum((y[mask] - y_pred) ** 2)
                    ss_tot = np.sum((y[mask] - y[mask].mean()) ** 2)
                    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

                    xpos = 0.03 if group == 'Oil Exporters' else 0.97
                    ha = 'left' if group == 'Oil Exporters' else 'right'
                    ypos = 0.95 if group == 'Oil Exporters' else 0.85
                    ax.text(xpos, ypos, f'R²={r2:.3f}', transform=ax.transAxes,
                            fontsize=7, ha=ha, color=color, fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                      edgecolor=color, alpha=0.7))

            ax.axhline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
            ax.axvline(0, color='grey', lw=0.4, ls='--', alpha=0.4)
            ax.grid(True, alpha=0.1)
            ax.tick_params(labelsize=7)

            if row == 0:
                ax.set_title(mname, fontsize=13, fontweight='bold')
            if col == 0:
                ax.set_ylabel(f'{period_label}\n\nCDS Spread (z)', fontsize=9)
            else:
                ax.set_ylabel('')
            if row == nrows - 1:
                ax.set_xlabel('Distance-to-Distress (z)', fontsize=9)
            else:
                ax.set_xlabel('')

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2,
               fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle('DtD vs CDS Scatter: Models × Regimes (z-scored)',
                 fontsize=14, fontweight='bold', y=1.05)
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    return fig


fig_scatter = plot_d2_scatter_grid(models, group_map,
                                    save_path='../output/figures/scatter_grid_appendix.png')
plt.show()

## A2 — Robustness: Spearman Correlations

In [ ]:
# ============================================================
# APPENDIX TABLE: SPEARMAN DiD
# ============================================================

print('Spearman ρ — Full Sample DiD')
print(f'{"Horizon":>8}', end='')
for mname in ['M0', 'M1', 'M2']:
    print(f'  |{mname + " Exp":>10}{mname + " Ctrl":>10}{mname + " DiD":>10}', end='')
print()
print('—' * 100)

for hlbl in HORIZONS:
    print(f'{hlbl:>8}', end='')
    for mname in ['M0', 'M1', 'M2']:
        gm = group_means(corr_results[mname]['Full Sample'], metric='rhoS')
        e = gm[hlbl]['exp_mean']
        c = gm[hlbl]['ctrl_mean']
        d = gm[hlbl]['did']
        print(f'  |{e:>10.4f}{c:>10.4f}{d:>+10.4f}', end='')
    print()

## A3 — Robustness: Directional Accuracy

In [ ]:
# ============================================================
# APPENDIX TABLE: DIRECTIONAL ACCURACY
# ============================================================

print('Directional Accuracy — Full Sample DiD')
print(f'{"Horizon":>8}', end='')
for mname in ['M0', 'M1', 'M2']:
    print(f'  |{mname + " Exp":>10}{mname + " Ctrl":>10}{mname + " DiD":>10}', end='')
print()
print('—' * 100)

for hlbl in HORIZONS:
    print(f'{hlbl:>8}', end='')
    for mname in ['M0', 'M1', 'M2']:
        gm = group_means(corr_results[mname]['Full Sample'], metric='Dir')
        e = gm[hlbl]['exp_mean']
        c = gm[hlbl]['ctrl_mean']
        d = gm[hlbl]['did']
        print(f'  |{e:>9.1%}{c:>10.1%}{d:>+10.1%}', end='')
    print()

## A4 — Country-Level Detail

In [ ]:
# ============================================================
# APPENDIX TABLE: FULL COUNTRY-LEVEL CORRELATIONS
# ============================================================

target_h = '1m'

print(f'Country-Level Pearson ρ ({target_h} horizon, Full Sample)')
print(f'{"Country":>18}{"Group":>14}', end='')
for mname in ['M0', 'M1', 'M2']:
    print(f'  |{mname:>12}', end='')
print(f'  |{"M1−M0":>8}{"M2−M0":>8}')
print('—' * 90)

# Collect data
country_rows = []
for country in sorted(group_map.keys()):
    row = {'country': country, 'group': group_map[country]}
    for mname in ['M0', 'M1', 'M2']:
        cdf = corr_results[mname]['Full Sample']
        c = cdf[cdf['Country'] == country]
        if len(c) > 0:
            row[f'{mname}_rho'] = c.iloc[0][f'rho_{target_h}']
            row[f'{mname}_p'] = c.iloc[0][f'p_{target_h}']
        else:
            row[f'{mname}_rho'] = np.nan
            row[f'{mname}_p'] = np.nan
    country_rows.append(row)

# Sort: exporters first, then controls
country_rows.sort(key=lambda r: (0 if r['group'] == 'Oil Exporter' else 1, r['country']))

prev_group = None
for row in country_rows:
    if row['group'] != prev_group and prev_group is not None:
        print('—' * 90)
    prev_group = row['group']

    print(f'{row["country"]:>18}{row["group"]:>14}', end='')
    for mname in ['M0', 'M1', 'M2']:
        rho = row[f'{mname}_rho']
        p = row[f'{mname}_p']
        if pd.notna(rho):
            print(f'  |{rho:>9.3f}{star(p):<3s}', end='')
        else:
            print(f'  |{"—":>12}', end='')

    # Improvement
    m0 = row['M0_rho']
    m1 = row['M1_rho']
    m2 = row['M2_rho']
    d1 = m1 - m0 if pd.notna(m1) and pd.notna(m0) else np.nan
    d2 = m2 - m0 if pd.notna(m2) and pd.notna(m0) else np.nan
    d1s = f'{d1:>+8.3f}' if pd.notna(d1) else f'{"—":>8}'
    d2s = f'{d2:>+8.3f}' if pd.notna(d2) else f'{"—":>8}'
    print(f'  |{d1s}{d2s}')

print('—' * 90)
print('Newey-West p-values. * p<0.10, ** p<0.05, *** p<0.01')